## Uncommon Runnables

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
BASE_URL = os.getenv('OPENAI_BASE_URL')

os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ["LANGCHAIN_TRACING_V2"] = "true" # Enable LangSmith tracing
os.environ['LANGSMITH_PROJECT'] = 'AgenticAITraining' # Set the LangSmith project name

In [2]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model='gpt-5.1',
    temperature=1.5,
    base_url=BASE_URL,
    api_key=API_KEY
)

In [37]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser, JsonOutputParser
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.runnables import RunnablePassthrough, RunnableBranch
import random
import time

#### Simple Extra

In [19]:
str_parser = StrOutputParser()

In [30]:
topic_template = ChatPromptTemplate.from_template(
    """
    Generate:
    1. A funny explanation of the topic
    2. A catchy headline

    Return response in JSON format with keys:
    content
    headline

    Topic: {topic}
    """
)

json_parser = JsonOutputParser()


tweet_template = ChatPromptTemplate.from_template(
    """
    Create an engaging tweet in questioning style.

    Headline:
    {headline}

    Content:
    {content}
    """
)

topic_chain = topic_template | llm | json_parser
tweet_chain = tweet_template | llm | str_parser

seq_chain = topic_chain | tweet_chain

### Lambda 
For running own funcion

In [4]:
from langchain_core.runnables import RunnableLambda

uppercase = RunnableLambda(lambda x: x.upper())
print(uppercase.invoke('hellop'))

HELLOP


In [5]:
# def This is for the summantion
def add(a, b):
    return a*2+b*3

In [6]:
random_sum = RunnableLambda(lambda x: add(x['a'], x['b']))
random_sum.invoke({'a':2, 'b':3})

13

### Passthorugh

Keep the original input while other runnables do extra work.


Don't do anything

In [15]:
passthrough = RunnablePassthrough()

print(passthrough.invoke("hello"))

hello


In [12]:
class Feedback(BaseModel):
    sentiment: Literal['positive', 'negative', 'neutral'] = Field(description="The sentiment of the feedback")

model = llm.with_structured_output(Feedback)

classification_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that classifies the sentiment of feedback."),
    ("user", "What is the sentiment of the following feedback? {feedback}")
])

classificaition_chain = classification_prompt | model

In [20]:
positive_template = ChatPromptTemplate([
    ("system", "You are a helpful assistant."),
    ("human","Generate a thank you note for this positive feedback: {feedback}."),
])

negative_template = ChatPromptTemplate([
    ("system", "You are a helpful assistant."),
    ("human", "Generate an apology note and offer assistance for this negative feedback: {feedback}."),
])

neutral_feedback_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "Generate a request for more details for this neutral feedback: {feedback}."),
])

default_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "Generate a response that escalates this feedback to a human representative: {feedback}."),
])


branch_chain = RunnableBranch(
    (lambda x: x.sentiment == 'positive', positive_template | llm | str_parser),
    (lambda x: x.sentiment == 'negative', negative_template | llm | str_parser),
    (lambda x: x.sentiment == 'neutral', neutral_feedback_prompt | llm | str_parser),
    default_template | llm | str_parser  # Default template
)

In [22]:
print(classificaition_chain)
print(branch_chain)

first=ChatPromptTemplate(input_variables=['feedback'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant that classifies the sentiment of feedback.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['feedback'], input_types={}, partial_variables={}, template='What is the sentiment of the following feedback? {feedback}'), additional_kwargs={})]) middle=[RunnableBinding(bound=ChatOpenAI(output_version=None, profile={'name': 'GPT-5.1', 'release_date': '2025-11-13', 'last_updated': '2025-11-13', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'struct

In [23]:
passthrough_chain = RunnablePassthrough()

chain = classificaition_chain | {
    'classificaion': passthrough_chain,
    'result': branch_chain
}

In [24]:
chain.get_graph().print_ascii()


            +-------------+              
            | PromptInput |              
            +-------------+              
                    *                    
                    *                    
                    *                    
         +--------------------+          
         | ChatPromptTemplate |          
         +--------------------+          
                    *                    
                    *                    
                    *                    
             +------------+              
             | ChatOpenAI |              
             +------------+              
                    *                    
                    *                    
                    *                    
               +--------+                
               | Lambda |                
               +--------+                
                    *                    
                    *                    
                    *             

In [25]:
result = chain.invoke({"feedback": "The product is okay. It works as expected but nothing exceptional."})

print(result)

{'classificaion': Feedback(sentiment='neutral'), 'result': 'Thanks for your neutral feedback. To help us understand your experience better, could you share a bit more detail?\n\n- What specifically worked well for you?\n- What didn’t work as well or felt “just okay”?\n- Was anything confusing, missing, or harder than you expected?\n- Is there one change or improvement that would have made your experience more positive?\n\nAny additional context or examples you can provide will help us improve.'}


In [26]:
print(type(result))
print(result.keys())
print(result['classificaion'])
print(type(result['classificaion']))
print(result['result'])

<class 'dict'>
dict_keys(['classificaion', 'result'])
sentiment='neutral'
<class '__main__.Feedback'>
Thanks for your neutral feedback. To help us understand your experience better, could you share a bit more detail?

- What specifically worked well for you?
- What didn’t work as well or felt “just okay”?
- Was anything confusing, missing, or harder than you expected?
- Is there one change or improvement that would have made your experience more positive?

Any additional context or examples you can provide will help us improve.


### Running Models in batches

In [31]:
seq_chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
  +--------------------+   
  | ChatPromptTemplate |   
  +--------------------+   
            *              
            *              
            *              
      +------------+       
      | ChatOpenAI |       
      +------------+       
            *              
            *              
            *              
  +------------------+     
  | JsonOutputParser |     
  +------------------+     
            *              
            *              
            *              
  +--------------------+   
  | ChatPromptTemplate |   
  +--------------------+   
            *              
            *              
            *              
      +------------+       
      | ChatOpenAI |       
      +------------+       
            *              
            *              
            *       

In [32]:
result = seq_chain.batch([
    {'topic': 'RAG in Finance'},
    {'topic': 'AI in Quant and Assest Management'},
    {'topic': 'AI in Education'}
])

In [34]:
print(len(result))
print(type(result[1]))
print(result[1])

3
<class 'langchain_core.messages.base.TextAccessor'>
“When Wall Street hires a robot intern that never sleeps, reads every 10-K ever written, runs 10M simulations, and calmly says ‘maybe don’t YOLO your retirement into meme stocks’… is AI becoming the quant’s new favorite nerd—or just an overconfident intern that still needs a human boss to say ‘no’ to the weird trades?” 🤖📉📈


### RunnableRetry
Becomes failproof, reties on failure

this is used with the model to retry

In [35]:
retry_model = ChatOpenAI(
    model='gpt-5.1',
    temperature=1.5,
    base_url=BASE_URL,
    api_key=API_KEY
).with_retry(
    stop_after_attempt=3
)

In [36]:
retry_model.invoke('Ping Hi how are you')

AIMessage(content='Hi! I’m here and ready to help. How are you doing today, and what would you like to work on or talk about?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 11, 'total_tokens': 49, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 5, 'engine_ttft_ms': 23, 'engine_ttlt_ms': 220, 'pre_inference_ms': 87, 'service_tbt_ms': 7, 'service_ttft_ms': 607, 'service_ttlt_ms': 861, 'total_duration_ms': 782, 'user_visible_ttft_ms': 520}}, 'model_provider': 'openai', 'model_name': 'gpt-5.1-2025-11-13', 'system_fingerprint': None, 'id': 'chatcmpl-DeJ72kQ4U4eOj5y3eePsyroXXc8nr', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e16cc-7725-7fe2-bc1b-fa386ac0dab6-0', tool_calls=[], invalid_tool

In [39]:
def fakey(x):
    random.seed(time.time())
    random_num = random.random()
    print(random_num)
    if random_num <0.6:
        print("FAILING")
        raise ValueError("Random failure")
    print('Passing')
    return x


fakeChain = RunnableLambda(fakey).with_retry(stop_after_attempt=5) | retry_model

fakeChain.batch(['Ping how are you', 'Ping how are you'])


0.8862088449004644
Passing
0.7274286994282424
Passing


[AIMessage(content='I don’t experience feelings, but I’m running normally and ready to help. What would you like to do today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 10, 'total_tokens': 44, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 8, 'engine_ttft_ms': 31, 'engine_ttlt_ms': 304, 'pre_inference_ms': 99, 'service_tbt_ms': 8, 'service_ttft_ms': 529, 'service_ttlt_ms': 787, 'total_duration_ms': 705, 'user_visible_ttft_ms': 430}}, 'model_provider': 'openai', 'model_name': 'gpt-5.1-2025-11-13', 'system_fingerprint': None, 'id': 'chatcmpl-DeJ8jN7E4eMNg4Pr6U8NxmmAN8F1F', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e16ce-11bf-70c2-b4e1-c6c72118d08d-0', tool_calls=[], invalid_tool

### Fallback Runnable

Changing the model in case of failures

In [40]:
# llm1 -> 5.1, llm2-> 4o
llm1 = ChatOpenAI(
    api_key = API_KEY,
    base_url = BASE_URL,
    model = 'gpt-5.1'
)
llm2 = ChatOpenAI(
    api_key = API_KEY,
    base_url = BASE_URL,
    model = 'gpt-4o'
)

print(llm1.invoke('ping'))
print(llm2.invoke('ping'))

content='pong' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 7, 'total_tokens': 18, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 3, 'engine_ttft_ms': 36, 'engine_ttlt_ms': 74, 'pre_inference_ms': 82, 'service_tbt_ms': 4, 'service_ttft_ms': 701, 'service_ttlt_ms': 737, 'total_duration_ms': 666, 'user_visible_ttft_ms': 620}}, 'model_provider': 'openai', 'model_name': 'gpt-5.1-2025-11-13', 'system_fingerprint': None, 'id': 'chatcmpl-DeLQBYIVtcFCbJF2Uw88PuG8AGemF', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019e1753-deb1-7c40-afc9-d5c6b7dc66e8-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 7, 'output_tokens': 11, 'total_tokens': 18, 'input_token_details': {'audio': 

In [ ]:
def custom_failure(x):
    r = random.random()
    print("random:", r)

    if r < 0.6:
        print("Failure")
        raise ValueError("Random Error")

    print("Pass")
    return x

def print_chain(x, name):
    print(name)
    return x

base_chain = (
    RunnableLambda(custom_failure)
    | RunnableLambda(lambda x: print_chain(x, 'base_chain'))  
)

backup_chain = (
    RunnableLambda(lambda x: print_chain(x, 'backup_chain'))
    | llm2
)

chain = base_chain.with_fallbacks([backup_chain])


In [52]:
chain.invoke('How are you')

random: 0.33370556663349127
Failure
backup_chain


AIMessage(content="I'm just a bunch of code, so I don't have feelings, but thank you for asking! 😊 How are you doing?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 10, 'total_tokens': 36, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 20, 'engine_ttft_ms': 146, 'engine_ttlt_ms': 664, 'pre_inference_ms': 88, 'service_tbt_ms': 20, 'service_ttft_ms': 548, 'service_ttlt_ms': 1065, 'total_duration_ms': 987, 'user_visible_ttft_ms': 460}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_af7f7349a4', 'id': 'chatcmpl-DeLZCz2ie4hAjQWwTRyWBU78pqhgX', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e175c-68c6-7892-873a-3481d33dda74-0', tool_calls=[], invali